# 类继承关系

```mermaid
classDiagram
    %% 元类层次结构
    class MetaStatsBuilderMixin {
        <<metaclass>>
    }
    class MetaPlotsBuilderMixin {
        <<metaclass>>
    }
    class MetaGenericAccessor {
        <<metaclass>>
    }
    class typing.Protocol {
        <<metaclass>>
    }
    class TransformerT
    class StatsBuilderMixin
    class PlotsBuilderMixin
    class BaseAccessor
    class GenericAccessor

    class BaseSRAccessor
    class BaseDFAccessor
    class GenericSRAccessor
    class GenericDFAccessor
    
    %% 继承关系
    typing.Protocol <|-- TransformerT

    MetaStatsBuilderMixin <|-- StatsBuilderMixin : metaclass
    MetaPlotsBuilderMixin <|-- PlotsBuilderMixin : metaclass
    MetaStatsBuilderMixin <|-- MetaGenericAccessor : metaclass
    MetaPlotsBuilderMixin <|-- MetaGenericAccessor : metaclass
    BaseAccessor <|-- GenericAccessor
    StatsBuilderMixin <|-- GenericAccessor
    PlotsBuilderMixin <|-- GenericAccessor
    MetaGenericAccessor <|-- GenericAccessor : metaclass

    BaseSRAccessor <|-- GenericSRAccessor
    GenericAccessor <|-- GenericSRAccessor
    

    GenericAccessor <|-- GenericDFAccessor
    BaseDFAccessor <|-- GenericDFAccessor
```

# class MetaGenericAccessor(type(StatsBuilderMixin), type(PlotsBuilderMixin))
通用访问器 `GenericAccessor` 的元类，结合了 `MetaStatsBuilderMixin` 和 `MetaPlotsBuilderMixin`，用于创建具有统计和绘图功能的访问器

```python
class MetaGenericAccessor(type(StatsBuilderMixin), type(PlotsBuilderMixin)):
    pass
```

# class TransformerT(tp.Protocol)
数据变换器的协议接口
- `transform`：对数据进行变换
- `fit_transform`：拟合并变换数据

```python
class TransformerT(tp.Protocol):
    def __init__(self, **kwargs) -> None:
        ...
    def transform(self, *args, **kwargs) -> tp.Array2d:
        ...
    def fit_transform(self, *args, **kwargs) -> tp.Array2d:
        ...
```

# `nb_config` 和 `transform_config`
`nb_config` 定义了要添加到 `GenericAccessor` 中的 numba 编译方法。

`transform_config` 定义了要添加到 `GenericAccessor` 中的 sklearn 风格数据变换方法。

```python
nb_config = Config(
    {
        # 数据洗牌 - 随机打乱数据顺序
        'shuffle': dict(func=nb.shuffle_nb, path='vectorbt.generic.nb.shuffle_nb'),
        # 填充 NaN 值
        'fillna': dict(func=nb.fillna_nb, path='vectorbt.generic.nb.fillna_nb'),
        # 向后移位 - 将数据向后移动指定位数
        'bshift': dict(func=nb.bshift_nb, path='vectorbt.generic.nb.bshift_nb'),
        # 向前移位 - 将数据向前移动指定位数
        'fshift': dict(func=nb.fshift_nb, path='vectorbt.generic.nb.fshift_nb'),
        # 差分计算 - 计算相邻期间的差值
        'diff': dict(func=nb.diff_nb, path='vectorbt.generic.nb.diff_nb'),
        # 百分比变化 - 计算相邻期间的百分比变化
        'pct_change': dict(func=nb.pct_change_nb, path='vectorbt.generic.nb.pct_change_nb'),
        # 向后填充 - 用后面的非 NaN 值填充 NaN
        'bfill': dict(func=nb.bfill_nb, path='vectorbt.generic.nb.bfill_nb'),
        # 向前填充 - 用前面的非 NaN 值填充 NaN
        'ffill': dict(func=nb.ffill_nb, path='vectorbt.generic.nb.ffill_nb'),
        # 累积和 - 计算忽略 NaN 的累积和
        'cumsum': dict(func=nb.nancumsum_nb, path='vectorbt.generic.nb.nancumsum_nb'),
        # 累积积 - 计算忽略 NaN 的累积积
        'cumprod': dict(func=nb.nancumprod_nb, path='vectorbt.generic.nb.nancumprod_nb'),
        # 滚动最小值 - 计算滚动窗口内的最小值
        'rolling_min': dict(func=nb.rolling_min_nb, path='vectorbt.generic.nb.rolling_min_nb'),
        # 滚动最大值 - 计算滚动窗口内的最大值
        'rolling_max': dict(func=nb.rolling_max_nb, path='vectorbt.generic.nb.rolling_max_nb'),
        # 滚动均值 - 计算滚动窗口内的平均值
        'rolling_mean': dict(func=nb.rolling_mean_nb, path='vectorbt.generic.nb.rolling_mean_nb'),
        # 扩展最小值 - 计算扩展窗口内的最小值
        'expanding_min': dict(func=nb.expanding_min_nb, path='vectorbt.generic.nb.expanding_min_nb'),
        # 扩展最大值 - 计算扩展窗口内的最大值
        'expanding_max': dict(func=nb.expanding_max_nb, path='vectorbt.generic.nb.expanding_max_nb'),
        # 扩展均值 - 计算扩展窗口内的平均值
        'expanding_mean': dict(func=nb.expanding_mean_nb, path='vectorbt.generic.nb.expanding_mean_nb'),
        # 乘积 - 计算忽略 NaN 的乘积（归约操作）
        'product': dict(func=nb.nanprod_nb, is_reducing=True, path='vectorbt.generic.nb.nanprod_nb')
    },
    readonly=True,    # 只读配置
    as_attrs=False    # 不作为属性访问
)
```

```python
transform_config = Config(
    {
        # 二值化 - 将数据转换为二进制形式
        'binarize': dict(
            transformer=Binarizer,
            docstring="参见 `sklearn.preprocessing.Binarizer`。"
        ),
        # 最小-最大缩放 - 将数据缩放到指定范围
        'minmax_scale': dict(
            transformer=MinMaxScaler,
            docstring="参见 `sklearn.preprocessing.MinMaxScaler`。"
        ),
        # 最大绝对值缩放 - 按最大绝对值缩放数据
        'maxabs_scale': dict(
            transformer=MaxAbsScaler,
            docstring="参见 `sklearn.preprocessing.MaxAbsScaler`。"
        ),
        # 标准化 - 将数据标准化为单位长度
        'normalize': dict(
            transformer=Normalizer,
            docstring="参见 `sklearn.preprocessing.Normalizer`。"
        ),
        # 鲁棒缩放 - 使用中位数和四分位数进行缩放
        'robust_scale': dict(
            transformer=RobustScaler,
            docstring="参见 `sklearn.preprocessing.RobustScaler`。"
        ),
        # 标准缩放 - 标准化为零均值单位方差
        'scale': dict(
            transformer=StandardScaler,
            docstring="参见 `sklearn.preprocessing.StandardScaler`。"
        ),
        # 分位数变换 - 将数据变换为均匀或正态分布
        'quantile_transform': dict(
            transformer=QuantileTransformer,
            docstring="参见 `sklearn.preprocessing.QuantileTransformer`。"
        ),
        # 幂变换 - 应用幂变换使数据更接近正态分布
        'power_transform': dict(
            transformer=PowerTransformer,
            docstring="参见 `sklearn.preprocessing.PowerTransformer`。"
        )
    },
    readonly=True,    # 只读配置
    as_attrs=False    # 不作为属性访问
)
```

# class GenericAccessor(BaseAccessor, StatsBuilderMixin, PlotsBuilderMixin, metaclass=MetaGenericAccessor)
```python
@attach_nb_methods(nb_config)
@attach_transform_methods(transform_config)
class GenericAccessor(BaseAccessor, StatsBuilderMixin, PlotsBuilderMixin, metaclass=MetaGenericAccessor): ...
```

通用数据访问器，适用于任何类型的数据，包括 `Series` 和 `DataFrame`，继承了多个混入类，集成了统计分析、绘图、基础数据操作等功能。

## `__init__`

```python
def __init__(self, obj: tp.SeriesFrame, mapping: tp.Optional[tp.MappingLike] = None, **kwargs) -> None:
    BaseAccessor.__init__(self, obj, mapping=mapping, **kwargs)
    StatsBuilderMixin.__init__(self)
    PlotsBuilderMixin.__init__(self)

    if mapping is not None:
        if isinstance(mapping, str):
            if mapping.lower() == 'index':
                mapping = self.wrapper.index
            elif mapping.lower() == 'columns':
                mapping = self.wrapper.columns
        mapping = to_mapping(mapping)
    self._mapping = mapping
```